# 00 はじめに — 基礎知識と用語から始めよう（画像のAI：検出・文字認識・システム化）

> 🔒 すべて**合成データ**（プログラムで作った、ナンバープレート風の画像）です。実在の車のナンバーは使いません。⏱ 所要時間: 読む 30分 ＋ 実行 数秒。**難しければ、「図の見方」と「用語集」だけ読んで、次のNotebookへ進んでください。**

予備知識ゼロから始めます。数式はほとんど使いません。分からない言葉が出たら、ここに戻ってきてください。

## 1. このプロジェクトは何をするの？（ひとことで）
**駐車場の入口カメラで撮った画像から、ナンバープレートを見つけ、文字を読み取り、入出庫の精算まで行う小さなシステム**を作ります。
ポイントは、AI単体の精度ではなく、**「AIは間違える」前提で、システム全体を安全に作る**ことです（自信がないときは人に任せる、など）。

流れは3つの仕事に分かれます。
1. **検出**: 画像のどこにナンバープレートがあるかを見つける。
2. **文字認識（OCR）**: 板の中の文字を、文字列として読む。
3. **システム化**: 読み間違いをルールで直し、自信が低ければ「要確認」にし、入庫の記録と照合して精算する。

## 2. 基礎知識

### 2-1. コンピュータにとって、画像とは
画像は、**小さな点（画素・ピクセル）の明るさを並べた数字の表**です。白黒画像なら、各画素が0（黒）〜255（白）の数字です。人間には「5」の文字に見えても、コンピュータには数字の並びにしか見えません。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

digit = np.array([[0, 0, 0, 0, 0, 0, 0, 0],
                  [0, 9, 9, 9, 9, 9, 0, 0],
                  [0, 9, 0, 0, 0, 0, 0, 0],
                  [0, 9, 9, 9, 9, 0, 0, 0],
                  [0, 0, 0, 0, 0, 9, 0, 0],
                  [0, 0, 0, 0, 0, 9, 0, 0],
                  [0, 9, 0, 0, 0, 9, 0, 0],
                  [0, 0, 9, 9, 9, 0, 0, 0]])
fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
ax[0].imshow(digit, cmap="gray_r"); ax[0].set_title("人間の目：数字の「5」に見える")
ax[1].imshow(digit, cmap="gray_r")
for (i, j), v in np.ndenumerate(digit):
    ax[1].text(j, i, int(v), ha="center", va="center", fontsize=8, color="red" if v else "gray")
ax[1].set_title("コンピュータ：数字の並び（9=黒い画素）")
plt.tight_layout(); plt.show()

**図の見方**: 左は8×8画素の画像（人間には「5」）。右は同じ画像を**数字で表した**もの。AIは、この数字の並びから「5だ」と判断することを学びます。実際の写真は、数十万〜数百万の画素と、色（赤緑青の3枚）でできています。

### 2-2. 「検出」— 枠で位置を示す（bbox と IoU）
物体の位置は、それをぴったり囲む**四角い枠（bbox：外接矩形）** で表します。AIの予測した枠が、正解の枠とどれだけ重なるかを **IoU**（重なりの割合）で測ります。IoU=1 で完全一致、0 で重なりなし。

In [ ]:
def iou(a, b):                       # 枠は [x, y, 幅, 高さ]
    ax1, ay1, ax2, ay2 = a[0], a[1], a[0] + a[2], a[1] + a[3]
    bx1, by1, bx2, by2 = b[0], b[1], b[0] + b[2], b[1] + b[3]
    w = max(0, min(ax2, bx2) - max(ax1, bx1)); h = max(0, min(ay2, by2) - max(ay1, by1))
    inter = w * h; union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union
truth = [20, 30, 60, 30]
for name, pred in [("ぴったり", [20, 30, 60, 30]), ("少しずれた", [24, 32, 60, 30]), ("大きくずれた", [50, 30, 60, 30]), ("全く別の場所", [100, 100, 60, 30])]:
    print(f"{name:8s} IoU = {iou(truth, pred):.2f}")
fig, ax = plt.subplots(figsize=(4.4, 3)); ax.set_xlim(0, 130); ax.set_ylim(140, 0)
for box, c, l in [(truth, "g", "正解"), ([24, 32, 60, 30], "r", "予測（少しずれた）")]:
    ax.add_patch(plt.Rectangle(box[:2], box[2], box[3], fill=False, ec=c, lw=2, label=l))
ax.legend(fontsize=8); ax.set_title("正解の枠と予測の枠"); plt.show()

**出力の見方**: ぴったりなら 1.00。少しずれただけでも 0.77 に下がり、大きくずれると 0.33、別の場所なら 0。このプロジェクトでは、**IoUが0.9以上**を「ちゃんと見つけた」の目安にしています（その後の文字認識に、十分な精度が必要なため）。

### 2-3. 「文字認識（OCR）」— 文字列を読む
OCR は、画像の中の文字を**文字列**にします。AIの出力が正しいかは、**CER（文字誤り率）** で測ります。**間違った文字の割合**で、0 なら全部正解です。

In [ ]:
def edit_distance(a, b):             # 文字の入れ替え・追加・削除が最少で何回かかるか
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]
truth = "品川 300 あ 12-34"
for pred in ["品川 300 あ 12-34", "品川 300 あ 12-84", "品川 800 あ 12-84", "品川 300 あ 1234"]:
    e = edit_distance(truth, pred)
    print(f"読んだ結果 {pred!r:22s} 間違い {e}文字 → CER = {e / len(truth):.2f}")

**出力の見方**: 1文字違うごとに CER は約 0.07 上がります（文字列の長さが14文字のため）。**1文字の読み間違いでも、精算は別の車のものになりかねない**ので、CER が小さいだけでなく、「**完全一致率**」（全部正解の割合）も重要です。

### 2-4. ニューラルネットワーク（CNN・RNN・CTC）を、ざっくりと
- **CNN**: 画像から、**線・角・形**などの特徴を、小さなフィルタで取り出すAI部品。画像の定番。
- **RNN／LSTM**: 並び（左から右へ読む文字列）を、**前後の文脈**を見ながら処理するAI部品。
- **CRNN**: CNNで特徴を取り、RNNで並びを読む。文字認識の定番の組み合わせ。
- **CTC**: 「どの文字が画像のどこにあるか」を教えなくても、**文字列だけを正解に**学習できる仕組み。
- **学習**: 正解付きの画像をたくさん見せ、**予測と正解のずれ（loss）が小さくなる方向に、少しずつ重みを直す**こと。1周見せるのを**エポック**と呼ぶ。
- **合成データの利点**: 学習のたびに新しい画像を作れて、正解が自動で付く。
- **転移学習**: 別の大量の画像で学習済みのAIを借りて、自分の課題に合わせて学習し直す。

### 2-5. 「確信度」と「要確認」— AIは間違える前提で
AIは、答えと一緒に**確信度**（どれくらい自信があるか）を出せます。確信度が低いときは、自動処理せずに**人に確認**します。しきい値の置き方で、2つの量が変わります。
- **自動処理率（カバレッジ）**: 自動で処理した割合。
- **自動処理の誤り率（リスク）**: 自動処理したもののうち、間違いだった割合。**これが最も重要**（誤課金に直結）。

In [ ]:
rng = np.random.default_rng(0)
n = 1000
correct = rng.random(n) < 0.9                                              # 90%は正しく読める
conf = np.where(correct, rng.beta(8, 1.5, n), rng.beta(3, 3, n))           # 正解のときは確信度が高めになりがち
print("しきい値  自動処理率  自動処理の誤り率")
for th in [0.0, 0.5, 0.7, 0.85, 0.95]:
    auto = conf >= th
    print(f"  {th:4.2f}     {auto.mean():5.1%}        {(~correct[auto]).mean():5.1%}")

**出力の見方**: しきい値0（全部自動）だと、1割が誤りです。しきい値を上げると、**自動処理する数は減る**（残りは人が確認）代わりに、**自動処理した中の誤りは大きく減る**。人手の手間と、誤りのリスクの**バランス**を決めるのが、運用の設計です。

### 2-6. ルールによる補正
ナンバーには**書式**があります（地名の一覧、桁数、使える文字）。AIの出力がこれに合わなければ、**書式に合う候補に差し替える**（補正）ことで、誤りを減らせます。これは機械学習ではなく、**人が書いたルール**です。

### 2-7. 入出庫システム
入庫時に「ナンバー・時刻」を**データベース**（記録の保管庫。ここでは軽量のSQLite）に保存。出庫時に読み取ったナンバーを入庫記録と**照合**して精算します。最悪の失敗は、**別の車の記録で課金する誤課金**です。

### 2-8. VLM — 学習なしで読むAI
**VLM**（画像と言語を扱えるAI）に、画像を見せて「ナンバーを読んで」と頼むと、**学習データを与えなくても**読めることがあります（**ゼロショット**）。自作のAIと比べて、「作る」か「借りる」かを考えます。画像を外部に送る場合は、プライバシーの検討が必要です。

## 3. 用語集（ここに戻ってきてください）

| 用語 | やさしい説明 |
|---|---|
| **画素（ピクセル）** | 画像を構成する小さな点。明るさ（や色）を数字で持つ。 |
| **合成データ・正解ラベル** | プログラムで作った練習用データ／画像に付ける正解の情報（板の位置、文字）。 |
| **検出・bbox** | 物体の位置を見つける／それを囲む四角い枠。 |
| **IoU** | 予測の枠と正解の枠の重なり（0〜1）。 |
| **四隅誤差** | 予測した板の四隅と、正解の四隅の距離（ピクセル）。 |
| **透視変換** | 斜めから見たようにゆがめる処理／傾いた板をまっすぐ直す処理。 |
| **OCR** | 画像の中の文字を文字列にすること。 |
| **CER・完全一致率** | 文字の間違いの割合／全部正解の割合。 |
| **古典的な画像処理** | 機械学習を使わず、人が書いたルールで画像を処理する方法。 |
| **CNN／RNN／CRNN／CTC** | 画像の特徴を取る部品／並びを読む部品／その組み合わせ／位置合わせ不要で文字列を学ぶ仕組み。 |
| **loss・エポック** | 予測と正解のずれ／学習データ1周。 |
| **転移学習** | 学習済みのAIを借りて、自分の課題に合わせて学び直すこと。 |
| **分布シフト・sim2real** | 学習時と使うときの条件が違うこと／合成と現実の違い。合成での精度は、現実で保証されない。 |
| **確信度・しきい値** | AIの自信の度合い／それ以上なら自動処理とする境目。 |
| **自動処理率・誤り率（リスク）** | 自動で処理した割合／その中の間違いの割合。 |
| **較正セット** | しきい値を決めるための、正解付きの別データ。 |
| **書式検査・補正** | 板の書式に合うか調べる／合うものに差し替える。 |
| **入庫・出庫・照合・誤課金** | 車が入る・出る／入庫の記録と突き合わせる／別の車の記録で課金してしまうこと。 |
| **VLM・ゼロショット・プロンプト** | 画像と言語を扱えるAI／学習データなしでいきなり解かせる／AIへの指示文。 |
| **信頼区間** | 標本が少ないとき「本当の値はこの範囲」という幅。差がこの幅に収まるなら誤差の範囲。 |
| **最終評価（test）** | 一度も調整に使っていないデータでの最後の評価。1回だけ。 |

## 4. このプロジェクトの地図（読む順）

| Notebook | 問い | 一言 |
|---|---|---|
| 00 合成データ | ナンバー風の画像をどう作る？ | 難易度を段階的に変える |
| 01 AIなしの基準 | ルールだけで、どこまで読める？ | 簡単な画像だけ |
| 02 検出 | AIで板の位置を見つけると？ | 例から学ぶ |
| 03 文字認識 | 文字列をAIで読むと？ | CRNN＋CTC |
| 04 ルール補正・確信度 | AIの間違いにどう備える？ | 要確認を作る |
| 05 入出庫システム | AIをシステムの部品にすると？ | 誤課金を避ける |
| 06 評価レポート | どこまで使える？ | 運用できる範囲がある |
| 07 VLM比較 | 学習なしのAIとの比較は？ | 作るか借りるか |

## 5. 結果を読むときの心得（5つ）
1. **基準（AIなし）と比べる。**
2. **「誤りの種類」を見る。** 平均の精度だけでなく、どの文字を取り違えるか。
3. **確信度が、正解と誤りを見分けられるかを確かめる。**
4. **合成データの精度は、現実では保証されない**（sim2real）。
5. **システム全体で評価する**（読み取り単体でなく、精算が正しいか）。

## 6. よくある疑問
- **Q. AIの精度が99%なら十分？** A. 1%の読み間違いが誤課金になるなら不十分です。だから確信度・要確認・照合のルールを重ねます。
- **Q. なぜ合成データ？** A. 実在の車のナンバー画像は個人情報になりうる。合成なら、正解も自動で付き、公開しても問題ありません。
- **Q. 数式が出てきたら？** A. 「出力の見方」の文章だけ追えば大丈夫です。

次は **00_synth_data** へ進みましょう。